**Nombre:** Alicia Dayana Pereira Tuqueres

# Ejercicio 9: Uso de la API de Google Gemini

En este ejercicio vamos a aprender a utilizar la API de OpenAI

## 1. Uso básico

El siguiente código sirve para conectarse con la API de Google Gemini de forma básica

In [1]:
!pip install python-dotenv

  Using cached python_dotenv-1.2.2-py3-none-any.whl.metadata (27 kB)
Using cached python_dotenv-1.2.2-py3-none-any.whl (22 kB)


In [2]:
import os
from google import genai
from dotenv import load_dotenv

# lee un archivo .env y carga la api key.
load_dotenv()

api_key = os.environ.get("GEMINI_API_KEY") 
client = genai.Client(api_key=api_key)

# Prueba de conexión básica
response = client.models.generate_content(
    model='gemini-2.5-flash',
    contents='Hola, ¿funciona la conexión?'
)
print(response.text)

c:\Users\Dayana\anaconda3\envs\examen_ri\Lib\site-packages\requests\__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


¡Hola! Sí, la conexión funciona perfectamente. ¿En qué puedo ayudarte hoy?


## 2. Retrieval

### 2.1 Cargo el corpus de 20 News Groups

Se usa `scikit-learn` para descargar rápidamente el dataset. En este caso, se limita la cantidad de documentos para que el proceso de embedding sea rápido y no consuma demasiados tokens.

In [ ]:
from sklearn.datasets import fetch_20newsgroups

# Descargar un subconjunto de categorías de tecnología/ciencia
newsgroups = fetch_20newsgroups(subset='train', categories=['sci.space', 'comp.graphics'])

# Limite de 50 docs
corpus = newsgroups.data[:50] 
print(f"Documentos cargados: {len(corpus)}")

Documentos cargados: 50


### 2.2 Transformo a embeddings

Se convierten los textos en vectores numéricos usando el modelo de embeddings de Gemini.

In [7]:
import numpy as np

print("Generar embeddings para el corpus...")
embeddings_documentos = []

for doc in corpus:
    res = client.models.embed_content(
        model='gemini-embedding-2',
        contents=doc,
    )
    embeddings_documentos.append(res.embeddings[0].values)

# Convertir la lista a un array de numpy para facilitar las operaciones matemáticas
embeddings_documentos = np.array(embeddings_documentos)
print("Embeddings generados exitosamente.")

Generar embeddings para el corpus...
Embeddings generados exitosamente.


### 2.3 Creo una query y hago la búsqueda

Se calcula la similitud coseno entre el embedding de la pregunta y los de los documentos del corpus para encontrar los más relevantes.

In [ ]:
query = "¿Cuáles son los avances en gráficos de computadora y renderizado 3D?"

res_query = client.models.embed_content(
    model='gemini-embedding-2', 
    contents=query,
)
# Redimensionar el vector de la query para la similitud coseno
embedding_query = np.array(res_query.embeddings[0].values).reshape(1, -1)


Obtengo los 5 documentos más similares a mi query

In [14]:
from sklearn.metrics.pairwise import cosine_similarity
# Calcular la similitud
similitudes = cosine_similarity(embedding_query, embeddings_documentos)[0]

# Obtener los 5 documentos más similares
top_k = 5
# np.argsort ordena de menor a mayor, usamos [::-1] para invertirlo y tomar los primeros 'top_k'
indices_top_k = np.argsort(similitudes)[::-1][:top_k]

documentos_relevantes = [corpus[i] for i in indices_top_k]

print(f"\nTop {top_k} documentos más similares a la query: '{query}'\n")
for i, doc in enumerate(documentos_relevantes):
    print(f"\nDocumento {i+1}:\n\n")
    # Imprimir solo los primeros 400 caracteres de cada documento para no saturar la salida
    print(doc[:400] + "...\n")


Top 5 documentos más similares a la query: '¿Cuáles son los avances en gráficos de computadora y renderizado 3D?'


Documento 1:


From: teckjoo@iti.gov.sg (Chua Teck Joo)
Subject: Visuallib (3D graphics for Windows)
Organization: Information Technology Institute, National Computer Board, Singapore.
Lines: 17


I am currently looking for a 3D graphics library that runs on MS
Windows 3.1.  Are there any such libraries out there other than
Visuallib?  (It must run on VGA and should not require any other
add-on graphics cards).
...


Documento 2:


From: zyeh@caspian.usc.edu (zhenghao yeh)
Subject: Re: Fast wireframe graphics
Organization: University of Southern California, Los Angeles, CA
Lines: 14
Distribution: usa
NNTP-Posting-Host: caspian.usc.edu


In article <C5tK4u.C6t@cs.columbia.edu>, ykim@cs.columbia.edu (Yong Su Kim) writes:
|> 
|> I am working on a program to display 3d wireframe models with the user
|> being able to arbitrarily ...


Documento 3:


From: chris@sarah.lerc.nas